# Water Stations Complete Analysis

## 🎯 Objective
Identify and extract water level (height) monitoring stations near our river erosion scope regions for use in predictive modeling.

## 📋 Deliverables
1. **`all_water_stations.gpkg`** - All water monitoring stations from RWS WFS service
2. **`water_stations_in_scope.gpkg`** - Stations within 100m of our scope regions
3. **`active_water_height_stations.gpkg`** - Active water height stations (2015+) in our scope regions
4. Reusable function for querying historical time series data

---

## 📚 The Discovery Journey

### Problem 1: Too Many Stations, Wrong Types
**Initial attempt:** Load `waterweb_locations.csv` (18,972 stations)
- **Issue:** This file contains ALL monitoring types (biology, chemistry, water quality, sediment, etc.)
- **Result:** Only ~0.08% had water level data when queried
- **Time cost:** Would take ~3 hours to query each station individually

### Discovery 1: WFS Service Exists! 🚀
RWS provides an OGC WFS service: `locatiesmetlaatstewaarneming`
- **Pre-filtered:** Only stations with recent confirmed measurements
- **Fast download:** CSV format, instant download (~30 seconds)
- **Rich metadata:** Parameter descriptions, latest measurement timestamps, coordinates
- **Total stations:** 840,055 (all types)

### Discovery 2: Filter by Parameter Description
**Key column:** `PARAMETER_WAT_OMSCHRIJVING`
- Filter for: `"Waterhoogte in Oppervlaktewater t.o.v. Normaal Amsterdams Peil in cm"`
- **Result:** 2,599 water height stations

### Discovery 3: Filter by Latest Measurement
**Key column:** `TIJDSTIP_LAATSTE_METING`
- Many stations are inactive (last measurement in 1880s, 2000s, 2013, etc.)
- Filter for: `>= 2015-01-01` to get truly active stations
- **Result:** 64 active water height stations in our scope regions

### Discovery 4: API Rate Limiting! ⚠️
**Problem:** Querying 2016-2025 (10 years) returns empty results
- **Root cause:** API has a volume limit per request
- **Evidence:**
  - 2023-2024 (2 years): ✅ Works (105k measurements)
  - 2016-2025 (10 years): ❌ Returns 204/empty
  - Only stations with <3 months of data succeeded in original query

### Solution: Chunked Queries (Year-by-Year)
Similar to `DataCollector` approach:
- Query each year individually (2016, 2017, ..., 2025)
- Concatenate results into single DataFrame
- **Time cost:** ~2-3 seconds per station for full 10-year history
- **Success rate:** ~100% for active stations

---

## 🔑 Key API Insights

### Waterweb API Endpoints
1. **`OphalenLaatsteWaarnemingen`** - Latest measurements only
2. **`OphalenWaarnemingen`** - Historical observations (use this!)
3. **`OphalenCatalogus`** - Parameter catalog

### Critical API Parameters
```json
{
  "Locatie": {"Code": "station_code"},
  "AquoPlusWaarnemingMetadata": {
    "AquoMetadata": {
      "Compartiment": {"Code": "OW"},  // Oppervlaktewater (surface water)
      "Grootheid": {"Code": "WATHTE"},  // Water height
      "ProcesType": "meting"  // CRITICAL: Only actual measurements (not predictions)
    }
  },
  "Periode": {
    "Begindatumtijd": "2023-01-01T00:00:00.000+01:00",
    "Einddatumtijd": "2023-12-31T23:59:59.000+01:00"
  }
}
```

### Station Code Column
The `CODE` column contains the station identifier needed for API queries.
Examples: `"steyl"`, `"venlo"`, `"deventer"`, `"zutphen.ijssel"`

---


## 1. Setup & Configuration

In [1]:
# Imports
import sys
sys.path.insert(0, "..")
sys.path.append("../..")

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from shapely.geometry import Point
from shapely import wkt
import requests
from datetime import datetime
from tqdm import tqdm
import time
import json
from io import StringIO

import src.paths as PATHS

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("✅ Imports successful!")

✅ Imports successful!


In [2]:
# Configuration
INPUT_GPKG = PATHS.DATA_DIR / "wocu_output_fase2_v4_w_wfs.gpkg"
SCOPE_LAYER = "vlakken_scope"
BUFFER_DISTANCE = 100  # meters - recommended based on analysis

# Output GeoPackages
OUTPUT_ALL_STATIONS = PATHS.DATA_DIR / "all_water_stations.gpkg"
OUTPUT_SCOPE_STATIONS = PATHS.DATA_DIR / "water_stations_in_scope.gpkg"
OUTPUT_ACTIVE_STATIONS = PATHS.DATA_DIR / "active_water_height_stations.gpkg"

# WFS Service (RWS)
WFS_BASE_URL = "https://geo.rijkswaterstaat.nl/services/ogc/hws/DDAPI20/ows"
WFS_PARAMS = {
    'SERVICE': 'WFS',
    'VERSION': '1.1.0',
    'REQUEST': 'GetFeature',
    'TYPENAME': 'locatiesmetlaatstewaarneming',  # Locations with latest measurements
    'outputFormat': 'csv',
    'srsName': 'EPSG:28992'  # Dutch RD New
}

# Waterweb API
BASE_URL = "https://ddapi20-waterwebservices.rijkswaterstaat.nl"
OBSERVATIONS_ENDPOINT = f"{BASE_URL}/ONLINEWAARNEMINGENSERVICES/OphalenWaarnemingen"
HEADERS = {
    "Content-Type": "application/json",
    "X-API-KEY": "dummy-key"
}

print(f"📂 Input: {INPUT_GPKG}")
print(f"📂 Output (all stations): {OUTPUT_ALL_STATIONS}")
print(f"📂 Output (scope stations): {OUTPUT_SCOPE_STATIONS}")
print(f"📂 Output (active stations): {OUTPUT_ACTIVE_STATIONS}")
print(f"🎯 Buffer distance: {BUFFER_DISTANCE}m")

📂 Input: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/wocu_output_fase2_v4_w_wfs.gpkg
📂 Output (all stations): /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/all_water_stations.gpkg
📂 Output (scope stations): /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/water_stations_in_scope.gpkg
📂 Output (active stations): /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/active_water_height_stations.gpkg
🎯 Buffer distance: 100m


## 2. Download All Water Stations from WFS Service

This downloads 840k+ monitoring stations with recent measurements from RWS.

In [3]:
print("🌐 Step 1: Downloading ALL water monitoring stations from RWS WFS...\n")
print(f"   Service: {WFS_BASE_URL}")
print(f"   Layer: {WFS_PARAMS['TYPENAME']}")
print(f"   This may take 30-60 seconds...\n")

try:
    response = requests.get(WFS_BASE_URL, params=WFS_PARAMS, timeout=120)
    response.raise_for_status()
    
    # Parse CSV
    all_stations_df = pd.read_csv(StringIO(response.text))
    
    print(f"✅ Downloaded {len(all_stations_df):,} stations\n")
    print(f"📊 Key columns:")
    print(f"   • CODE: Station identifier")
    print(f"   • NAAM: Station name")
    print(f"   • PARAMETER_WAT_OMSCHRIJVING: What is measured")
    print(f"   • TIJDSTIP_LAATSTE_METING: Latest measurement timestamp")
    print(f"   • GEOMETRY: Point location (WKT format)\n")
    
    print(f"📋 Sample data:")
    print(all_stations_df[['CODE', 'NAAM', 'PARAMETER_WAT_OMSCHRIJVING', 'TIJDSTIP_LAATSTE_METING']].head(10))
    
except Exception as e:
    print(f"❌ Failed to download: {e}")
    raise

🌐 Step 1: Downloading ALL water monitoring stations from RWS WFS...

   Service: https://geo.rijkswaterstaat.nl/services/ogc/hws/DDAPI20/ows
   Layer: locatiesmetlaatstewaarneming
   This may take 30-60 seconds...



/var/folders/t3/519_jjmx4tz2_jlt9nm2gkpr0000gn/T/ipykernel_65392/4213757678.py:11: DtypeWarning: Columns (0: GROEPERINGCODE) have mixed types. Specify dtype option on import or set low_memory=False.
  all_stations_df = pd.read_csv(StringIO(response.text))


✅ Downloaded 840,064 stations

📊 Key columns:
   • CODE: Station identifier
   • NAAM: Station name
   • PARAMETER_WAT_OMSCHRIJVING: What is measured
   • TIJDSTIP_LAATSTE_METING: Latest measurement timestamp
   • GEOMETRY: Point location (WKT format)

📋 Sample data:
                         CODE                          NAAM  \
0  grijpskerk.gaarkeuken.oost    Grijpskerk Gaarkeuken oost   
1      dordrecht.oudemaas.120       Dordrecht Oude Maas 120   
2                europlatform                  Europlatform   
3        dalfsen.vechterweerd         Dalfsen, Vechterweerd   
4              europlatform.3                Europlatform 3   
5              negenoord.oost               Negenoord, oost   
6                    berkhout                      Berkhout   
7  ijmuiden.noordersluis.west  IJmuiden, noordersluis, west   
8              nieuwvossemeer               Nieuw-Vossemeer   
9              nieuwvossemeer               Nieuw-Vossemeer   

                          PARAMETER_WA

### Convert to GeoDataFrame and Save

In [4]:
print("🗺️ Converting to GeoDataFrame...\n")

# Parse WKT geometry
all_stations_gdf = gpd.GeoDataFrame(
    all_stations_df,
    geometry=all_stations_df['GEOMETRY'].apply(wkt.loads),
    crs='EPSG:28992'  # RD New - Dutch national grid
)

# Drop FID column (GeoPackage creates its own)
if 'FID' in all_stations_gdf.columns:
    all_stations_gdf = all_stations_gdf.drop(columns=['FID'])

print(f"✅ Created GeoDataFrame")
print(f"   Stations: {len(all_stations_gdf):,}")
print(f"   CRS: {all_stations_gdf.crs}")
print(f"   Bounds: {all_stations_gdf.total_bounds}\n")

# Save to GeoPackage #1
print(f"💾 Saving to: {OUTPUT_ALL_STATIONS.name}...")
all_stations_gdf.to_file(OUTPUT_ALL_STATIONS, layer='all_monitoring_stations', driver='GPKG')
print(f"✅ Saved! ({len(all_stations_gdf):,} stations)")

🗺️ Converting to GeoDataFrame...

✅ Created GeoDataFrame
   Stations: 840,064
   CRS: EPSG:28992
   Bounds: [-8803488.9998072   -356245.0667026    614737.1859015   1465854.84632442]

💾 Saving to: all_water_stations.gpkg...
✅ Saved! (840,064 stations)


## 3. Filter for Water Height Stations

From 840k+ stations, filter to only those measuring water height ("Waterhoogte").

In [5]:
print("🔍 Step 2: Filtering for WATER HEIGHT stations...\n")

# Analyze parameter types
print("📊 Top 15 parameter types in dataset:\n")
param_counts = all_stations_gdf['PARAMETER_WAT_OMSCHRIJVING'].value_counts()
print(param_counts.head(15).to_string())

# Filter for water height
water_height_keyword = "Waterhoogte in Oppervlaktewater t.o.v. Normaal Amsterdams Peil in cm"
water_height_stations = all_stations_gdf[
    all_stations_gdf['PARAMETER_WAT_OMSCHRIJVING'] == water_height_keyword
].copy()

print(f"\n" + "="*80)
print(f"WATER HEIGHT FILTERING RESULTS")
print(f"="*80)
print(f"Total stations (all types): {len(all_stations_gdf):,}")
print(f"Water height stations: {len(water_height_stations):,} ({len(water_height_stations)/len(all_stations_gdf)*100:.2f}%)")
print(f"\n💡 We now have {len(water_height_stations):,} stations that measure water level!")

🔍 Step 2: Filtering for WATER HEIGHT stations...

📊 Top 15 parameter types in dataset:

PARAMETER_WAT_OMSCHRIJVING
Zuurgraad in Oppervlaktewater                                                                          95178
(massa)Concentratie zuurstof in Oppervlaktewater in mg/l                                               86403
Temperatuur in Oppervlaktewater in oC                                                                  83560
Geleidendheid in Oppervlaktewater in mS/m                                                              81783
Verzadigingsgraad zuurstof in Oppervlaktewater in %                                                    80996
Saliniteit in Oppervlaktewater                                                                         79216
(massa)Concentratie chlorofyl-a in Oppervlaktewater in ug/l                                            73951
Waterhoogte in Oppervlaktewater t.o.v. Normaal Amsterdams Peil in cm                                    2291
Aanwezigheid 

## 4. Load Scope Regions & Perform Spatial Analysis

Find which water height stations are within 100m of our scope regions.

In [6]:
print(f"📥 Step 3: Loading scope regions...\n")

scope_regions = gpd.read_file(INPUT_GPKG, layer=SCOPE_LAYER)

print(f"✅ Loaded {len(scope_regions):,} regions")
print(f"   CRS: {scope_regions.crs}")
print(f"   Bounds: {scope_regions.total_bounds}")

# Find ID column
id_col = None
for col in ['location_id', 'position_id', 'region_id', 'id', 'FID']:
    if col in scope_regions.columns:
        id_col = col
        break

if id_col:
    print(f"   ID column: {id_col}")
else:
    print(f"   ⚠️ No ID column found, will use index")
    scope_regions['region_id'] = scope_regions.index
    id_col = 'region_id'

📥 Step 3: Loading scope regions...

✅ Loaded 12,130 regions
   CRS: EPSG:28992
   Bounds: [102109.45629229 320046.33231488 212124.74522418 517091.22026353]
   ID column: position_id


In [7]:
print(f"\n🔍 Performing spatial join (buffer={BUFFER_DISTANCE}m)...\n")

# Ensure same CRS
if water_height_stations.crs != scope_regions.crs:
    print(f"   Reprojecting stations to {scope_regions.crs}")
    water_height_stations = water_height_stations.to_crs(scope_regions.crs)

# Buffer scope regions
print(f"   Creating {BUFFER_DISTANCE}m buffer around regions...")
buffered_regions = scope_regions.copy()
buffered_regions['geometry'] = scope_regions.geometry.buffer(BUFFER_DISTANCE)

# Spatial join
print(f"   Performing spatial join...")
stations_in_scope = gpd.sjoin(
    water_height_stations,
    buffered_regions[[id_col, 'geometry']],
    how='inner',
    predicate='intersects'
)

# Get unique stations
unique_stations_in_scope = water_height_stations[
    water_height_stations['CODE'].isin(stations_in_scope['CODE'].unique())
].copy()

# Get coverage statistics
regions_with_stations = stations_in_scope[id_col].nunique()
coverage_pct = (regions_with_stations / len(scope_regions)) * 100

print(f"\n" + "="*80)
print(f"SPATIAL ANALYSIS RESULTS ({BUFFER_DISTANCE}m buffer)")
print(f"="*80)
print(f"Water height stations in area: {len(unique_stations_in_scope):,}")
print(f"Scope regions covered: {regions_with_stations:,}/{len(scope_regions):,} ({coverage_pct:.1f}%)")
print(f"Station-region pairs: {len(stations_in_scope):,}")


🔍 Performing spatial join (buffer=100m)...

   Creating 100m buffer around regions...
   Performing spatial join...

SPATIAL ANALYSIS RESULTS (100m buffer)
Water height stations in area: 642
Scope regions covered: 858/12,130 (7.1%)
Station-region pairs: 3,073


### Save Water Stations in Scope

In [8]:
print(f"\n💾 Saving to: {OUTPUT_SCOPE_STATIONS.name}...")

# Drop FID if exists
if 'FID' in unique_stations_in_scope.columns:
    unique_stations_in_scope = unique_stations_in_scope.drop(columns=['FID'])

# Save scope regions
scope_regions.to_file(OUTPUT_SCOPE_STATIONS, layer='scope_regions', driver='GPKG')
print(f"   ✅ Saved layer: scope_regions ({len(scope_regions):,} regions)")

# Save stations
unique_stations_in_scope.to_file(OUTPUT_SCOPE_STATIONS, layer=f'water_height_stations_{BUFFER_DISTANCE}m', driver='GPKG')
print(f"   ✅ Saved layer: water_height_stations_{BUFFER_DISTANCE}m ({len(unique_stations_in_scope):,} stations)")

print(f"\n✅ GeoPackage #2 complete!")


💾 Saving to: water_stations_in_scope.gpkg...
   ✅ Saved layer: scope_regions (12,130 regions)
   ✅ Saved layer: water_height_stations_100m (642 stations)

✅ GeoPackage #2 complete!


## 5. Filter for ACTIVE Stations (2015+)

Many stations are inactive. Filter for those with recent measurements.

In [9]:
print(f"🔍 Step 4: Filtering for ACTIVE stations (latest measurement >= 2015)...\n")

# Convert timestamp to datetime
unique_stations_in_scope['TIJDSTIP_LAATSTE_METING'] = pd.to_datetime(
    unique_stations_in_scope['TIJDSTIP_LAATSTE_METING']
)

# Show distribution of latest measurement dates
print(f"📊 Latest measurement date distribution:\n")
unique_stations_in_scope['last_measurement_year'] = unique_stations_in_scope['TIJDSTIP_LAATSTE_METING'].dt.year
year_counts = unique_stations_in_scope['last_measurement_year'].value_counts().sort_index()
print(year_counts.tail(15).to_string())

# Filter for active (2015+)
cutoff_date = pd.Timestamp('2015-01-01', tz='UTC')
active_stations = unique_stations_in_scope[
    unique_stations_in_scope['TIJDSTIP_LAATSTE_METING'] >= cutoff_date
].copy()

print(f"\n" + "="*80)
print(f"ACTIVE STATION FILTERING")
print(f"="*80)
print(f"Stations in scope area: {len(unique_stations_in_scope):,}")
print(f"Active stations (2015+): {len(active_stations):,} ({len(active_stations)/len(unique_stations_in_scope)*100:.1f}%)")
print(f"\n💡 These {len(active_stations)} stations have recent water level measurements!")

# List active stations
print(f"\n📋 Active station codes:\n")
active_station_codes = sorted(active_stations['CODE'].unique())
for i, code in enumerate(active_station_codes, 1):
    print(f"   {i:2d}. {code}")

🔍 Step 4: Filtering for ACTIVE stations (latest measurement >= 2015)...

📊 Latest measurement date distribution:

last_measurement_year
2006    20
2007     8
2008     1
2009    22
2010    12
2011     3
2012     9
2013    92
2014     1
2017     2
2018     1
2019     1
2024     4
2025    70
2026    70

ACTIVE STATION FILTERING
Stations in scope area: 642
Active stations (2015+): 148 (23.1%)

💡 These 148 stations have recent water level measurements!

📋 Active station codes:

    1. amerongen.beneden
    2. amerongen.boven
    3. arnhem.nederrijn
    4. belfeld.boven
    5. buggenum
    6. culemborg
    7. dalem
    8. desteeg
    9. desteeg.haven
   10. deventer
   11. dodewaard
   12. doesburg.ijssel
   13. driel.beneden
   14. driel.boven
   15. eisdenmazenhove.maas
   16. eisdenmazenhove.maesbempdergreend
   17. elsloo.maas
   18. genemuiden
   19. gennep
   20. grave.beneden
   21. grave.boven
   22. grevenbicht
   23. hagestein.beneden
   24. hagestein.boven
   25. hank.bergschemaas

### Save Active Water Height Stations

In [10]:
print(f"\n💾 Saving to: {OUTPUT_ACTIVE_STATIONS.name}...")

# Drop FID if exists
if 'FID' in active_stations.columns:
    active_stations = active_stations.drop(columns=['FID'])

# Add metadata columns
active_stations['is_active_2015plus'] = True
active_stations['buffer_distance_m'] = BUFFER_DISTANCE

# Save scope regions
scope_regions.to_file(OUTPUT_ACTIVE_STATIONS, layer='scope_regions', driver='GPKG')
print(f"   ✅ Saved layer: scope_regions ({len(scope_regions):,} regions)")

# Save active stations
active_stations.to_file(OUTPUT_ACTIVE_STATIONS, layer='active_water_height_stations', driver='GPKG')
print(f"   ✅ Saved layer: active_water_height_stations ({len(active_stations):,} stations)")

print(f"\n✅ GeoPackage #3 complete!")


💾 Saving to: active_water_height_stations.gpkg...
   ✅ Saved layer: scope_regions (12,130 regions)
   ✅ Saved layer: active_water_height_stations (148 stations)

✅ GeoPackage #3 complete!


## 6. Historical Time Series Query Function

### The API Rate Limiting Discovery

**Problem Found:** The Waterweb API has a volume limit per request!

Evidence:
- Query 2023-2024 (2 years): ✅ Returns 105,000+ measurements
- Query 2016-2025 (10 years): ❌ Returns 204 (No Content)
- Query 2020-2024 (5 years): ❌ Returns 204 (No Content)

**Solution:** Query year-by-year and concatenate (similar to `DataCollector` approach)

In [11]:
def fetch_water_level_data_chunked(station_code, start_year=2016, end_year=2025):
    """
    Fetch water level measurements for a station, querying year-by-year to avoid API limits.
    
    Args:
        station_code: Station identifier from CODE column
        start_year: First year to query (default: 2016)
        end_year: Last year to query (default: 2025)
    
    Returns:
        DataFrame with columns: timestamp, water_level_cm, station_code, year
        Returns None if no data available
    """
    all_dfs = []
    
    for year in range(start_year, end_year + 1):
        start_date = f"{year}-01-01T00:00:00.000+01:00"
        end_date = f"{year}-12-31T23:59:59.000+01:00"
        
        body = {
            "Locatie": {"Code": station_code},
            "AquoPlusWaarnemingMetadata": {
                "AquoMetadata": {
                    "Compartiment": {"Code": "OW"},  # Oppervlaktewater
                    "Grootheid": {"Code": "WATHTE"},  # Water height
                    "ProcesType": "meting"  # CRITICAL: Only actual measurements!
                }
            },
            "Periode": {
                "Begindatumtijd": start_date,
                "Einddatumtijd": end_date
            }
        }
        
        try:
            response = requests.post(
                OBSERVATIONS_ENDPOINT,
                json=body,
                headers=HEADERS,
                timeout=60
            )
            
            if response.status_code == 204:
                continue  # No data for this year
            
            response.raise_for_status()
            data = response.json()
            
            if "WaarnemingenLijst" in data and data["WaarnemingenLijst"]:
                for obs_series in data["WaarnemingenLijst"]:
                    if "MetingenLijst" in obs_series and obs_series["MetingenLijst"]:
                        measurements = obs_series["MetingenLijst"]
                        
                        records = []
                        for m in measurements:
                            record = {
                                'timestamp': m.get('Tijdstip'),
                                'water_level_cm': m.get('Meetwaarde', {}).get('Waarde_Numeriek'),
                                'station_code': station_code
                            }
                            records.append(record)
                        
                        if records:
                            df = pd.DataFrame(records)
                            df['timestamp'] = pd.to_datetime(df['timestamp'])
                            df['year'] = df['timestamp'].dt.year
                            all_dfs.append(df)
                            
        except Exception as e:
            # Continue with other years even if one fails
            continue
        
        # Small delay to be nice to the API
        time.sleep(0.2)
    
    # Combine all years
    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        return combined_df
    else:
        return None

print("✅ Water level query function defined: fetch_water_level_data_chunked()")
print("\n💡 Usage example:")
print("   df = fetch_water_level_data_chunked('steyl', start_year=2020, end_year=2024)")
print("   Time cost: ~2-3 seconds per station for 10 years of data")

✅ Water level query function defined: fetch_water_level_data_chunked()

💡 Usage example:
   df = fetch_water_level_data_chunked('steyl', start_year=2020, end_year=2024)
   Time cost: ~2-3 seconds per station for 10 years of data


### Test the Function

In [12]:
# Test with one active station
test_station = active_station_codes[0]
print(f"🧪 Testing data query with station: {test_station}\n")
print(f"   Querying 2023-2024 (2 years)...")

test_df = fetch_water_level_data_chunked(test_station, start_year=2023, end_year=2024)

if test_df is not None:
    print(f"\n✅ SUCCESS!")
    print(f"   Measurements: {len(test_df):,}")
    print(f"   Date range: {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")
    print(f"   Years: {sorted(test_df['year'].unique())}")
    
    # Show yearly breakdown
    yearly = test_df.groupby('year').size()
    print(f"\n   Measurements per year:")
    for year, count in yearly.items():
        print(f"      {year}: {count:,}")
    
    print(f"\n📊 Sample data:")
    print(test_df.head(10))
else:
    print(f"❌ No data returned for {test_station}")

🧪 Testing data query with station: amerongen.beneden

   Querying 2023-2024 (2 years)...

✅ SUCCESS!
   Measurements: 105,216
   Date range: 2023-01-01 00:00:00+01:00 to 2024-12-31 23:50:00+01:00
   Years: [np.int32(2023), np.int32(2024)]

   Measurements per year:
      2023: 52,560
      2024: 52,656

📊 Sample data:
                  timestamp  water_level_cm       station_code  year
0 2023-01-01 00:00:00+01:00           412.0  amerongen.beneden  2023
1 2023-01-01 00:10:00+01:00           412.0  amerongen.beneden  2023
2 2023-01-01 00:20:00+01:00           412.0  amerongen.beneden  2023
3 2023-01-01 00:30:00+01:00           412.0  amerongen.beneden  2023
4 2023-01-01 00:40:00+01:00           412.0  amerongen.beneden  2023
5 2023-01-01 00:50:00+01:00           412.0  amerongen.beneden  2023
6 2023-01-01 01:00:00+01:00           413.0  amerongen.beneden  2023
7 2023-01-01 01:10:00+01:00           412.0  amerongen.beneden  2023
8 2023-01-01 01:20:00+01:00           413.0  amerongen.bene

## 7. Summary & Next Steps

In [ ]:
print("="*80)
print("WATER STATIONS ANALYSIS - COMPLETE SUMMARY")
print("="*80)

print(f"\n📊 Data Processing Pipeline:")
print(f"   1. Downloaded from WFS: {len(all_stations_gdf):,} monitoring stations (all types)")
print(f"   2. Filtered for water height: {len(water_height_stations):,} stations")
print(f"   3. Spatial filter ({BUFFER_DISTANCE}m): {len(unique_stations_in_scope):,} stations near scope regions")
print(f"   4. Active filter (2015+): {len(active_stations):,} stations with recent data")

print(f"\n📦 Output Files Created:")
print(f"   1. {OUTPUT_ALL_STATIONS.name}")
print(f"      • all_monitoring_stations ({len(all_stations_gdf):,} stations)")
print(f"\n   2. {OUTPUT_SCOPE_STATIONS.name}")
print(f"      • scope_regions ({len(scope_regions):,} regions)")
print(f"      • water_height_stations_{BUFFER_DISTANCE}m ({len(unique_stations_in_scope):,} stations)")
print(f"\n   3. {OUTPUT_ACTIVE_STATIONS.name}")
print(f"      • scope_regions ({len(scope_regions):,} regions)")
print(f"      • active_water_height_stations ({len(active_stations):,} stations)")

print(f"\n🎯 Coverage Statistics:")
print(f"   • {regions_with_stations:,}/{len(scope_regions):,} regions have nearby water stations ({coverage_pct:.1f}%)")
print(f"   • {len(active_station_codes)} stations available for time series queries")

print(f"\n🔑 Key Technical Learnings:")
print(f"   • WFS service provides pre-filtered stations (much faster than CSV)")
print(f"   • Filter by 'PARAMETER_WAT_OMSCHRIJVING' column for water height")
print(f"   • Filter by 'TIJDSTIP_LAATSTE_METING' >= 2015 for active stations")
print(f"   • API queries MUST be chunked (year-by-year) to avoid volume limits")
print(f"   • Use 'ProcesType': 'meting' in API body for actual measurements only")
print(f"   • 'CODE' column contains station identifier for API queries")

print(f"\n🚀 Next Steps for DataHandler Integration:")
print(f"   1. Load 'active_water_height_stations' layer from GeoPackage #3")
print(f"   2. For each scope region, find nearest station(s) within {BUFFER_DISTANCE}m")
print(f"   3. Use fetch_water_level_data_chunked() to get time series")
print(f"   4. Extract features:")
print(f"      • High water events (>95th percentile)")
print(f"      • Maximum water level")
print(f"      • Frequency of high water events")
print(f"      • Water level variability (std dev)")
print(f"   5. For regions without nearby stations:")
print(f"      • Use nearest neighbor interpolation")
print(f"      • Or use regional average values")
print(f"      • Or use normalized/zero values with a flag")

print(f"\n" + "="*80)
print(f"✅ ANALYSIS COMPLETE")
print(f"="*80)

## Appendix: Example - Query Multiple Stations

In [ ]:
# Example: Query first 3 active stations for 2023-2024
# Uncomment to run

# print(f"📊 Example: Querying first 3 stations for 2023-2024...\n")

# all_data = []
# for station_code in active_station_codes[:3]:
#     print(f"   Querying: {station_code}...")
#     df = fetch_water_level_data_chunked(station_code, start_year=2023, end_year=2024)
#     if df is not None:
#         print(f"      ✅ {len(df):,} measurements")
#         all_data.append(df)
#     else:
#         print(f"      ❌ No data")

# if all_data:
#     combined = pd.concat(all_data, ignore_index=True)
#     print(f"\n✅ Combined data: {len(combined):,} measurements from {len(all_data)} stations")
#     print(f"   Date range: {combined['timestamp'].min()} to {combined['timestamp'].max()}")
#     
#     # Calculate high water threshold (95th percentile)
#     threshold = combined['water_level_cm'].quantile(0.95)
#     high_water = combined[combined['water_level_cm'] > threshold]
#     print(f"\n💧 High water analysis (>95th percentile):")
#     print(f"   Threshold: {threshold:.1f} cm")
#     print(f"   High water events: {len(high_water):,} ({len(high_water)/len(combined)*100:.1f}%)")